In [0]:
input_path = "/Volumes/workspace/default/stream_demo/input"
bronze_path = "/Volumes/workspace/default/stream_demo/bronze"
silver_path = "/Volumes/workspace/default/stream_demo/silver"


bronze_checkpoint = "/Volumes/workspace/default/stream_demo/checkpoints/bronze"
silver_checkpoint = "/Volumes/workspace/default/stream_demo/checkpoints/silver"

# Nothing runs yet.

In [0]:
# Streaming requires schema defined first.

from pyspark.sql.types import *

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("amount", DoubleType(), True)
])

In [0]:
#Monitors folder continuously
# Whenever new files arrive, Spark reads them automatically.

orders_stream = spark.readStream.format("csv").option("header" , True).schema(schema).load(input_path)

In [0]:


bronze_query = orders_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", bronze_checkpoint) \
    .trigger(availableNow=True)\
    .start(bronze_path)

# Pipeline:
# input folder
#     ↓
# Spark reads files
#     ↓
# Data written to Bronze Delta table

# Stored in:
# /Volumes/workspace/default/stream_demo/bronze

# Checkpoint creation
# Spark automatically creates:

# /Volumes/.../checkpoints/bronze

In [0]:
# Check results:

spark.read.format("delta").load(bronze_path).show()

In [0]:
# Silver Streaming Transformation

# Read Bronze as a stream.
bronze_stream = spark.readStream \
    .format("delta") \
    .load(bronze_path)

# Apply transformation:
silver_df = bronze_stream.filter("amount > 500")



In [0]:
silver_query = silver_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation" , silver_checkpoint) \
    .trigger(availableNow=True) \
    .start(silver_path)


In [0]:
# View Bronze table:
spark.read.format("delta").load(bronze_path).show()

# View Silver table:
spark.read.format("delta").load(silver_path).show()

In [0]:


# Streaming runs continuously.
bronze_query.stop()
silver_query.stop()